## Qwen3 fine-tuning on SQuARE-Wikidata — generic contrastive recipe

Fine-tunes **`Qwen/Qwen3-Embedding-0.6B`** on the **SQuARE** benchmark
(result-set-grounded SPARQL query pairs), adapting the backbone with a
**generic, off-the-shelf contrastive objective** — in-batch-negatives
InfoNCE, the same recipe used to fine-tune most open sentence-embedding
models (e.g. `sentence-transformers`' `MultipleNegativesRankingLoss`):

    candidates = concat(P, N)                     # [2B, d]: B positives + B dataset negatives
    logits     = scale * (A @ candidates.T)        # [B, 2B], scale = temperature^-1 = 20 (matches SparqlCE's own τ=0.05)
    labels     = arange(B)                         # anchor i's positive sits at column i
    loss       = CrossEntropy(logits, labels)       # standard in-batch-negatives InfoNCE
    lr         = argmax_val-MRR over {5e-5, 1e-4, 2e-4} (short probe, Section 7)
    epochs     = early-stopped on val MRR, patience 3, budget <= 12 (Section 8)
    optimizer  = AdamW, cosine schedule with warmup, grad-clip 1.0
    selection  = best val MRR (embedding ranking, same protocol as SparqlCE)
    splits     = paper's fixed split (splits_wikidata.json), seeded fallback otherwise

**Why this objective, and why it still isn't SparqlCE.** For a fair
baseline, Qwen3 needs to see the *same dataset* SparqlCE was trained and
credited on — withholding the dataset from baselines would be the actual
fairness violation, not a way to protect our contribution. What must stay
ours is the *training design*: SparqlCE is not "contrastive learning" in
the abstract, it is InfoNCE **plus** a graded Jaccard-calibration
regression term, a graph encoder over parsed SPARQL structure, and a
variable-renaming augmentation curriculum. This notebook therefore uses
the most generic, textbook contrastive recipe available — plain
in-batch-negatives InfoNCE over raw text, with no Jaccard calibration, no
hard-negative mining beyond the dataset's own annotated negatives, and no
structural augmentation — so any remaining gap to SparqlCE can be
attributed to our specific design rather than to "we used a contrastive
loss and you didn't."

**Matched against SparqlCE's own recipe, not asserted.** Two things that
used to be picked by hand are now tuned/verified the same way SparqlCE's
own hyperparameters were: the learning rate is chosen by a short sweep
(Section 7) rather than one guessed value, and the epoch budget is decided
by early stopping on val MRR (Section 8) rather than a fixed guess ("LoRA
converges faster"). The in-batch negative pool size (32: SparqlCE's
`BATCH_SIZE=32` vs. this notebook's 16 positives + 16 dataset negatives per
step) and temperature/scale (SparqlCE `τ=0.05` == `SCALE=20` here) are
already matched.

This notebook is the Wikidata run; see `04_*` for the DBpedia counterpart
(identical recipe, different `KG` default and dataset paths).

**What is revised vs. SparqlCE:**

| Component        | SparqlCE (CodeBERT + GATv2)                  | This notebook (Qwen3)                          |
|------------------|----------------------------------------------|------------------------------------------------|
| Input            | Query parsed into a graph (`SPARQLGraphBuilder`) | Raw SPARQL text, tokenized directly        |
| Encoder          | CodeBERT token emb + GATv2 + mean pool + proj | Qwen3-Embedding decoder, **last-token pooling** |
| Objective        | Contrastive InfoNCE + Jaccard-calibration MSE | **Generic in-batch-negatives InfoNCE only**    |
| Negatives        | Graded (Jaccard-weighted) + in-batch          | **Binary dataset negatives + in-batch, unweighted** |
| Negative pool     | 32 (batch size)                              | 32 (16 in-batch positives + 16 dataset negatives) |
| Temperature/scale | τ = 0.05                                     | scale = 20 (== 1/0.05)                         |
| LR selection      | Differential LR, tuned over method development | **Swept** over {5e-5, 1e-4, 2e-4} by val MRR (Section 7) |
| Epoch budget      | 20 (fixed)                                    | **Early-stopped** on val MRR, patience 3, budget <= 12 (Section 8) |
| Trainable params | Full model, differential LRs (1e-5 / 1e-4)    | **LoRA adapters only**, LR chosen by sweep     |
| Precision        | fp32                                          | bf16 backbone + gradient checkpointing         |
| Augmentation     | Random variable renaming (graph-side)         | None (standard baseline)                       |

Split handling, ranking-based validation, and checkpoint-selection logic are
carried over from SparqlCE so the two models are directly comparable; only
the training *objective* (and the design elements listed above) differ, by
design.

Run top-to-bottom. Verify the paths in the CONFIG cell first.


In [ ]:
# Dependencies (uncomment on a fresh environment)
# %pip install -q torch transformers>=4.51 peft accelerate tqdm


In [1]:
import os
import sys
import json
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset, random_split
from transformers import AutoModel, AutoTokenizer, get_cosine_schedule_with_warmup
from peft import LoraConfig, get_peft_model, PeftModel
from tqdm.auto import tqdm

# ============================================================================
# 1. CONFIG — verify these paths match your repo before running
# ============================================================================
KG = "wikidata"   # "dbpedia" | "wikidata"

PATHS = {
    "dbpedia": {
        "data":   "../../datasets/dbpedia-triplets-dataset.json",
        "splits": "../../datasets/splits_dbpedia.json",
    },
    "wikidata": {
        "data":   "../../datasets/wikidata-triplets-dataset.json",
        "splits": "../../datasets/splits_wikidata.json",
    },
}
DATA_PATH   = PATHS[KG]["data"]
SPLITS_PATH = PATHS[KG]["splits"]
CKPT_DIR    = f"../../checkpoints/qwen3_square_{KG}"

MODEL_NAME  = "Qwen/Qwen3-Embedding-0.6B"

# Generic (non-SparqlCE) contrastive fine-tuning recipe: plain
# in-batch-negatives InfoNCE on explicit (anchor, positive, negative)
# triples. SparqlCE-specific components (Jaccard calibration, graph
# encoder, augmentation curriculum) are DELIBERATELY EXCLUDED so this
# baseline credits only a generic, textbook contrastive adaptation of a
# strong LLM embedder on SQuARE (see src-code/BASELINE_REDESIGN.md).
SEED         = 42
WEIGHT_DECAY = 1e-2
GRAD_CLIP    = 1.0

# Epoch budget: NOT a fixed guess. MAX_EPOCHS is just an upper bound; the
# training loop (Section 8) early-stops on val MRR with patience PATIENCE,
# so the reported model is trained to its own convergence, not to an
# arbitrarily small epoch count.
MAX_EPOCHS   = 12
PATIENCE     = 3

# Qwen3-specific settings
BATCH_SIZE   = 16            # per-step; each anchor yields one pos + one neg pair
                             # -> 32 contrastive candidates/step, matching SparqlCE's
                             # own BATCH_SIZE=32 in-batch-negative pool size
GRAD_ACCUM   = 2             # effective batch 32, matches the SparqlCE recipe
SCALE        = 20.0          # InfoNCE temperature scale == 1 / SparqlCE's τ=0.05
MAX_LEN      = 512           # tokens per SPARQL query
WARMUP_FRAC  = 0.05
AUGMENT_RENAME_PROB = 0.0    # no domain-specific augmentation in the standard baseline

# Learning rate: NOT a single hand-picked value. Section 7 sweeps these
# candidates with a short probe run and keeps whichever gets the best val
# MRR, so the baseline is tuned to the same standard as SparqlCE's own
# (differential) LR, which reflects choices made over method development.
LR_CANDIDATES   = [5e-5, 1e-4, 2e-4]
LR_SWEEP_EPOCHS = 2

LORA_R       = 16
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05
LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj",
                "gate_proj", "up_proj", "down_proj"]


/upb/users/z/zahera/profiles/unix/cs/.conda/envs/sparql-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ============================================================================
# 2. Setup: repo import path, seeds, device
# ============================================================================
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

PROJECT_ROOT = str(Path.cwd().parent.parent)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Reuse the variable-renaming augmentation from the SparqlCE dataset module so
# both models see the same invariance signal during training.
from sparqlce.dataset import _random_rename_variables  # noqa: E402

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device = {device}")


device = cuda


/upb/users/z/zahera/profiles/unix/cs/.conda/envs/sparql-env/lib/python3.10/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: Could not load this library: /local/upb/users/z/zahera/profiles/unix/cs/.conda/envs/sparql-env/lib/python3.10/site-packages/torch_scatter/_version_cuda.so
  import torch_geometric.typing
/upb/users/z/zahera/profiles/unix/cs/.conda/envs/sparql-env/lib/python3.10/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-cluster'. Disabling its usage. Stacktrace: Could not load this library: /local/upb/users/z/zahera/profiles/unix/cs/.conda/envs/sparql-env/lib/python3.10/site-packages/torch_cluster/_version_cuda.so
  import torch_geometric.typing
/upb/users/z/zahera/profiles/unix/cs/.conda/envs/sparql-env/lib/python3.10/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-spline-conv'. Disa

### 3. Data

Same SQuARE JSON schema as `SemanticSPARQLDataset`
(`anchor_query` / `positive_queries` / `negative_queries`), but the dataset
below is **text-only**: no graph builder, no PyG batching. Each item yields
an explicit `(anchor_text, positive_text, negative_text)` triple — one
positive pair and one dataset negative per anchor per step, used as the
labelled positive/negative in the in-batch InfoNCE loss (Section 6). Unlike
the SparqlCE recipe, the graded `answer_jaccard` values are not used
anywhere: this baseline only sees binary match / non-match membership, and
weights every negative (in-batch or dataset-provided) equally.


In [3]:
class SquareTextDataset(Dataset):
    """Anchor-level SQuARE dataset over raw SPARQL text.

    Text-side revision of ``sparqlce.dataset.SemanticSPARQLDataset``: same
    JSON parsing and augmentation, minus the graph construction and minus
    the graded Jaccard signal (only used here to pick match / non-match
    pairs used as the labelled positive/negative in the InfoNCE loss).
    """

    def __init__(self, datapath, augment_rename_prob=0.0, augment_seed=None):
        self.augment_rename_prob = float(augment_rename_prob)
        self._aug_rng = random.Random(augment_seed) if augment_seed is not None else random
        self.groups = []

        print(f"Loading dataset from {datapath}...")
        with open(datapath, "r", encoding="utf-8") as f:
            raw = json.load(f)

        for item in raw:
            anchor = item.get("anchor_query")
            if not anchor:
                continue
            pos_list = [
                p["query"] for p in item.get("positive_queries", [])
                if p.get("query") is not None
            ]
            neg_list = [
                n["query"] for n in item.get("negative_queries", [])
                if n.get("query") is not None
            ]
            if not pos_list or not neg_list:
                continue
            self.groups.append({"anchor": anchor, "positives": pos_list, "negatives": neg_list})

        print(f"Loaded {len(self.groups)} anchor groups.")

    def _maybe_augment(self, query):
        if self.augment_rename_prob > 0.0 and self._aug_rng.random() < self.augment_rename_prob:
            return _random_rename_variables(query, self._aug_rng)
        return query

    def __len__(self):
        return len(self.groups)

    def __getitem__(self, idx):
        g = self.groups[idx]
        pos_q = random.choice(g["positives"])
        neg_q = random.choice(g["negatives"])
        return (self._maybe_augment(g["anchor"]),
                self._maybe_augment(pos_q),
                self._maybe_augment(neg_q))


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, padding_side="left")

def collate_pairs(batch):
    """Tokenize (anchor, positive, negative) triples into three padded batches."""
    anchors, positives, negatives = zip(*batch)
    a_tok = tokenizer(list(anchors),   padding=True, truncation=True,
                      max_length=MAX_LEN, return_tensors="pt")
    p_tok = tokenizer(list(positives), padding=True, truncation=True,
                      max_length=MAX_LEN, return_tensors="pt")
    n_tok = tokenizer(list(negatives), padding=True, truncation=True,
                      max_length=MAX_LEN, return_tensors="pt")
    return a_tok, p_tok, n_tok


In [4]:
# ============================================================================
# 3b. Fixed split (paper) with safe random fallback — unchanged from SparqlCE
# ============================================================================
dataset = SquareTextDataset(
    DATA_PATH,
    augment_rename_prob=AUGMENT_RENAME_PROB,
    augment_seed=SEED,
)
# Validation/test must see un-augmented queries: share the file, not the flags.
eval_dataset = SquareTextDataset(DATA_PATH, augment_rename_prob=0.0)

if os.path.exists(SPLITS_PATH):
    with open(SPLITS_PATH, "r") as f:
        splits = json.load(f)
    train_idx = splits.get("train_idx", splits.get("train"))
    val_idx   = splits.get("val_idx",   splits.get("valid_idx", splits.get("val")))
    test_idx  = splits.get("test_idx",  splits.get("test"))
    assert train_idx and val_idx and test_idx, f"Could not parse splits from {SPLITS_PATH}"

    train_dataset = Subset(dataset,      train_idx)
    val_dataset   = Subset(eval_dataset, val_idx)
    test_dataset  = Subset(eval_dataset, test_idx)
    print(f"Using FIXED split from {SPLITS_PATH}")
else:
    print(f"WARNING: fixed split file not found at {SPLITS_PATH}")
    print("WARNING: falling back to seeded random_split — numbers will NOT match the paper.")
    total   = len(dataset)
    n_train = int(0.80 * total)
    n_val   = int(0.10 * total)
    n_test  = total - n_train - n_val
    gen = torch.Generator().manual_seed(SEED)
    train_split, val_split, test_split = random_split(
        range(total), [n_train, n_val, n_test], generator=gen)
    train_dataset = Subset(dataset,      list(train_split.indices))
    val_dataset   = Subset(eval_dataset, list(val_split.indices))
    test_dataset  = Subset(eval_dataset, list(test_split.indices))

print(f"train={len(train_dataset)}  val={len(val_dataset)}  test={len(test_dataset)}")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate_pairs, drop_last=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=collate_pairs)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=collate_pairs)


Loading dataset from ../../datasets/wikidata-triplets-dataset.json...
Loaded 10735 anchor groups.
Loading dataset from ../../datasets/wikidata-triplets-dataset.json...
Loaded 10735 anchor groups.
Using FIXED split from ../../datasets/splits_wikidata.json
train=8588  val=1073  test=1074


### 4. Model: Qwen3-Embedding + LoRA

Qwen3-Embedding is a decoder-only model: the query embedding is the hidden
state of the **last non-padding token** (the pooling recommended on the model
card), L2-normalised — this replaces SparqlCE's GATv2 + mean-pool + projection
head. Only LoRA adapters are trained; the 0.6B backbone stays frozen in bf16.

No classification head, no projection head beyond the backbone's own
hidden state: the pooled, L2-normalised embedding produced by `forward` is
used directly for both the InfoNCE training loss (Section 6) and inference
(Sections 7-10), via plain cosine similarity / dot product.

`build_model_and_optim(lr, epochs, steps_per_epoch)` builds a **fresh**
model + optimizer + scheduler for a given learning rate — used once per
candidate in the Section 7 LR sweep, and once more for the final Section 8
training run at the LR the sweep selected.


In [5]:
def last_token_pool(last_hidden_states, attention_mask):
    """Last-token pooling as on the Qwen3-Embedding model card.
    Handles both left- and right-padded batches."""
    left_padding = (attention_mask[:, -1].sum() == attention_mask.shape[0])
    if left_padding:
        return last_hidden_states[:, -1]
    seq_lens = attention_mask.sum(dim=1) - 1
    idx = torch.arange(last_hidden_states.shape[0], device=last_hidden_states.device)
    return last_hidden_states[idx, seq_lens]


class Qwen3SparqlEmbedder(nn.Module):
    """Qwen3-Embedding backbone + LoRA, producing L2-normalised embeddings.

    Embedding output has the same contract as ``HybridSPARQLEncoder.forward``
    (unit-norm float embeddings), so the SparqlCE evaluation code applies
    verbatim. There is no auxiliary head: the same ``forward`` output is
    used for training (InfoNCE, Section 6) and inference (Sections 7-10).
    """

    def __init__(self, model_name, lora_cfg=None, adapter_path=None):
        super().__init__()
        base = AutoModel.from_pretrained(
            model_name,
            torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        )
        base.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs={"use_reentrant": False})
        base.enable_input_require_grads()  # needed for checkpointing + frozen base
        if adapter_path is not None:
            self.backbone = PeftModel.from_pretrained(base, adapter_path)
        else:
            self.backbone = get_peft_model(base, lora_cfg)

    def forward(self, tok):
        out = self.backbone(input_ids=tok["input_ids"],
                            attention_mask=tok["attention_mask"])
        emb = last_token_pool(out.last_hidden_state, tok["attention_mask"])
        return F.normalize(emb.float(), p=2, dim=1)


lora_cfg = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGETS,
    bias="none",
    task_type="FEATURE_EXTRACTION",
)


def build_model_and_optim(lr, epochs, steps_per_epoch):
    """Fresh model + optimizer + scheduler at a given LR/epoch budget.

    Used once per candidate LR in the Section 7 sweep (short probe runs)
    and once more for the real Section 8 training run — so every LR is
    evaluated under identical conditions instead of being picked by hand.
    """
    m = Qwen3SparqlEmbedder(MODEL_NAME, lora_cfg=lora_cfg).to(device)
    opt = torch.optim.AdamW(
        (p for p in m.parameters() if p.requires_grad),
        lr=lr, weight_decay=WEIGHT_DECAY,
    )
    total_steps = max((steps_per_epoch // GRAD_ACCUM) * epochs, 1)
    sched = get_cosine_schedule_with_warmup(
        opt,
        num_warmup_steps=int(WARMUP_FRAC * total_steps),
        num_training_steps=total_steps,
    )
    return m, opt, sched


# One-off print of trainable-parameter count (same for every LR candidate).
_probe, _, _ = build_model_and_optim(LR_CANDIDATES[0], 1, len(train_loader))
_probe.backbone.print_trainable_parameters()
del _probe
if torch.cuda.is_available():
    torch.cuda.empty_cache()


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 310/310 [00:00<00:00, 1002.13it/s, Materializing param=norm.weight]                              


trainable params: 10,092,544 || all params: 605,869,056 || trainable%: 1.6658


In [6]:
# ============================================================================
# 5. Loss — generic in-batch-negatives InfoNCE (NOT SparqlCE's loss).
#
# This is the standard contrastive recipe used by most open sentence-
# embedding fine-tunes (equivalent to sentence-transformers'
# MultipleNegativesRankingLoss with explicit hard negatives appended): for
# each anchor, its true positive is scored against every positive AND every
# dataset negative in the batch, softmax-normalised. No graded-similarity
# (Jaccard) regression term, no calibration, no hard-negative mining beyond
# the dataset's own annotated negatives, no graph encoder — those are
# SparqlCE's contribution and are deliberately excluded here
# (see BASELINE_REDESIGN.md).
# ============================================================================
def info_nce_loss(a_emb, p_emb, n_emb, scale):
    """In-batch-negatives InfoNCE: anchor i's positive is column i of
    concat(P, N); every other positive and every dataset negative in the
    batch acts as an additional, unweighted negative."""
    candidates = torch.cat([p_emb, n_emb], dim=0)          # [2B, d]
    logits = scale * (a_emb @ candidates.T)                # [B, 2B]
    labels = torch.arange(a_emb.size(0), device=a_emb.device)
    loss = F.cross_entropy(logits, labels)
    with torch.no_grad():
        acc = (logits.argmax(dim=1) == labels).float().mean()
    return loss, acc


In [7]:
# ============================================================================
# 6. Validation: rank each anchor's true positive against ALL positives in
# the split, using cosine similarity of the pooled embeddings only (no
# auxiliary head is involved — the same embeddings used for training).
# Identical protocol to SparqlCE — selects the embedding that retrieves
# correctly, not the one that collapses everything to a single point.
# ============================================================================
@torch.no_grad()
def evaluate_ranking(model, loader, device):
    model.eval()
    anchors, positives = [], []
    for a_tok, p_tok, _ in loader:
        a_tok = {k: v.to(device) for k, v in a_tok.items()}
        p_tok = {k: v.to(device) for k, v in p_tok.items()}
        anchors.append(model(a_tok).cpu())
        positives.append(model(p_tok).cpu())
    A = torch.cat(anchors)      # [N, d]
    P = torch.cat(positives)    # [N, d]
    sims  = A @ P.T             # [N, N]; diagonal is the true positive
    diag  = sims.diag().unsqueeze(1)
    ranks = (sims >= diag).sum(dim=1).float()   # 1-indexed rank of true positive
    return {
        "mrr":    (1.0 / ranks).mean().item(),
        "hits@1": (ranks == 1).float().mean().item(),
        "hits@5": (ranks <= 5).float().mean().item(),
    }


In [ ]:
# ============================================================================
# 7. Learning-rate sweep — pick LR by short-probe val MRR, not by guesswork.
#
# SparqlCE's own differential LR (1e-5 / 1e-4) reflects choices made over
# the course of developing the method. Picking one LR for the baseline by
# hand would hold it to a lower standard, so instead each candidate here is
# probed for LR_SWEEP_EPOCHS epochs and the one with the best val MRR is
# carried into the full Section 8 training run.
# ============================================================================
print(f"LR sweep over {LR_CANDIDATES} ({LR_SWEEP_EPOCHS} probe epochs each)")
sweep_results = []

for lr in LR_CANDIDATES:
    torch.manual_seed(SEED)
    probe_model, probe_opt, probe_sched = build_model_and_optim(
        lr, LR_SWEEP_EPOCHS, len(train_loader))

    for epoch in range(1, LR_SWEEP_EPOCHS + 1):
        probe_model.train()
        probe_opt.zero_grad()
        pbar = tqdm(train_loader, desc=f"lr={lr:.0e} probe epoch {epoch}/{LR_SWEEP_EPOCHS}")
        for step, (a_tok, p_tok, n_tok) in enumerate(pbar, start=1):
            a_tok = {k: v.to(device) for k, v in a_tok.items()}
            p_tok = {k: v.to(device) for k, v in p_tok.items()}
            n_tok = {k: v.to(device) for k, v in n_tok.items()}

            a_emb = probe_model(a_tok)
            p_emb = probe_model(p_tok)
            n_emb = probe_model(n_tok)
            loss, _ = info_nce_loss(a_emb, p_emb, n_emb, SCALE)
            (loss / GRAD_ACCUM).backward()

            if step % GRAD_ACCUM == 0:
                torch.nn.utils.clip_grad_norm_(
                    (p for p in probe_model.parameters() if p.requires_grad), max_norm=GRAD_CLIP)
                probe_opt.step()
                probe_sched.step()
                probe_opt.zero_grad()

    probe_val = evaluate_ranking(probe_model, val_loader, device)
    sweep_results.append({"lr": lr, "val_mrr": probe_val["mrr"]})
    print(f"  lr={lr:.0e} -> val MRR {probe_val['mrr']:.4f}")

    del probe_model, probe_opt, probe_sched
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

best_lr = max(sweep_results, key=lambda r: r["val_mrr"])["lr"]
print(f"\nSelected LR = {best_lr:.0e} (best {LR_SWEEP_EPOCHS}-epoch probe val MRR)")


LR sweep over [5e-05, 0.0001, 0.0002] (2 probe epochs each)


lr=5e-05 probe epoch 2/2: 100%|██████████| 536/536 [14:27<00:00,  1.62s/it]


  lr=5e-05 -> val MRR 0.4151


lr=1e-04 probe epoch 1/2:  20%|██        | 108/536 [02:55<10:02,  1.41s/it]

In [ ]:
# ============================================================================
# 8. Training loop — generic in-batch-negatives InfoNCE fine-tuning at the
# swept LR, full budget with early stopping (gradient accumulation). Trains
# up to MAX_EPOCHS, stopping once val MRR hasn't improved for PATIENCE
# epochs — so the epoch count is decided by convergence, not asserted.
# ============================================================================
os.makedirs(CKPT_DIR, exist_ok=True)
best_mrr  = -1.0
best_path = os.path.join(CKPT_DIR, "best_adapter")
history   = []
epochs_without_improvement = 0

model, optimizer, scheduler = build_model_and_optim(best_lr, MAX_EPOCHS, len(train_loader))
model.backbone.print_trainable_parameters()

print(f"Training Qwen3 on SQuARE-{KG} for up to {MAX_EPOCHS} epochs "
      f"(lr={best_lr:.0e}, effective batch {BATCH_SIZE * GRAD_ACCUM}, patience={PATIENCE})")
print("loss = InfoNCE(A, [P; N]) with in-batch negatives — "
      "generic contrastive, no Jaccard calibration, no graph")

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    sum_loss, sum_acc, n_steps = 0.0, 0.0, 0
    optimizer.zero_grad()
    pbar = tqdm(train_loader, desc=f"Epoch {epoch:02d}/{MAX_EPOCHS}")

    for step, (a_tok, p_tok, n_tok) in enumerate(pbar, start=1):
        a_tok = {k: v.to(device) for k, v in a_tok.items()}
        p_tok = {k: v.to(device) for k, v in p_tok.items()}
        n_tok = {k: v.to(device) for k, v in n_tok.items()}

        a_emb = model(a_tok)
        p_emb = model(p_tok)
        n_emb = model(n_tok)

        loss, acc = info_nce_loss(a_emb, p_emb, n_emb, SCALE)
        (loss / GRAD_ACCUM).backward()

        if step % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(
                (p for p in model.parameters() if p.requires_grad), max_norm=GRAD_CLIP)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        sum_loss += loss.item(); sum_acc += acc.item(); n_steps += 1
        pbar.set_postfix(loss=f"{loss.item():.4f}", acc=f"{acc.item():.4f}")

    val_metrics = evaluate_ranking(model, val_loader, device)
    history.append({"epoch": epoch,
                    "train_loss": sum_loss / n_steps,
                    "train_acc": sum_acc / n_steps,
                    **{f"val_{k}": v for k, v in val_metrics.items()}})
    print(f"Epoch {epoch:02d} | loss {sum_loss/n_steps:.4f} "
          f"acc {sum_acc/n_steps:.4f} | "
          f"val MRR {val_metrics['mrr']:.4f} "
          f"H@1 {val_metrics['hits@1']:.4f} H@5 {val_metrics['hits@5']:.4f}")

    # Selection by val MRR (embedding-space ranking), as in the SparqlCE
    # recipe. Only the LoRA adapter is saved (~a few MB).
    if val_metrics["mrr"] > best_mrr:
        best_mrr = val_metrics["mrr"]
        epochs_without_improvement = 0
        model.backbone.save_pretrained(best_path)
        tokenizer.save_pretrained(best_path)
        print(f"  \u2713 new best val MRR {best_mrr:.4f} \u2014 adapter saved to {best_path}")
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print(f"  val MRR hasn't improved for {PATIENCE} epochs \u2014 stopping early "
                  f"at epoch {epoch} (best val MRR {best_mrr:.4f}).")
            break


In [ ]:
# ============================================================================
# 9. Sanity check on the training-file test split (pair-ranking metric)
#
# NOTE: this is only the quick diagonal-ranking check used during training.
# The numbers reported in the paper's RQ1 tables come from Section 10
# below, which follows the exact protocol of
# notebooks/evaluation-RQ1/evaluate_rq1.py.
# ============================================================================
print(f"Training complete. Best val MRR: {best_mrr:.4f} (lr={best_lr:.0e})")
print("Reloading best adapter and evaluating on the test split...")

del model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

best_model = Qwen3SparqlEmbedder(MODEL_NAME, adapter_path=best_path).to(device)
test_metrics = evaluate_ranking(best_model, test_loader, device)

print(f"\nSQuARE-{KG} test sanity check (fine-tuned Qwen3-Embedding-0.6B):")
for k, v in test_metrics.items():
    print(f"  {k:>7}: {v:.4f}")

with open(os.path.join(CKPT_DIR, "test_metrics.json"), "w") as f:
    json.dump({"kg": KG, "model": MODEL_NAME, "best_lr": best_lr, "best_val_mrr": best_mrr,
               "test": test_metrics, "history": history,
               "lr_sweep": sweep_results}, f, indent=2)
print(f"\nMetrics written to {os.path.join(CKPT_DIR, 'test_metrics.json')}")


### 10. RQ1 evaluation on the held-out test set (paper tables)

This is the evaluation that fills the **"Qwen3-Emb-0.6B (fine-tuned)"** row in
the paper's RQ1 tables (`tables/tab-RQ1-DBpedia.tex` / `tab-RQ1-Wikidata.tex`).
It follows the **exact** protocol of `notebooks/evaluation-RQ1/evaluate_rq1.py`,
the same script used for all baselines and SparqlCE:

- **Data:** `datasets/{kg}_testset_RQ1.json` (DBpedia: 337 anchors, 1677 corpus
  queries) — *not* the training-file split used above for model selection.
- **Retrieval:** each anchor is ranked against the **global corpus** (all unique
  anchor/positive/negative queries in the test set), anchor self-match excluded.
- **Metrics:** MRR, Hits, nDCG (ranking) and P, R, F1 (set overlap) at
  k = 3, 5, 10, 20, averaged over anchors with ≥1 annotated positive.

The only model-specific part is the encoder: queries are embedded with the
fine-tuned Qwen3 model (best-val-MRR LoRA adapter from the LR the Section 7
sweep selected, last-token pooling, L2-normalised), mirroring
`build_emb_map_sentence_transformer` in the script.


In [ ]:
# ============================================================================
# 10a. Load the RQ1 test set and build corpus / eval items / relevance maps
#      — exactly as notebooks/evaluation-RQ1/{DBpedia,Wikidata}/06_eval_*.ipynb
# ============================================================================
_RQ1_DIR = str(Path(PROJECT_ROOT) / "notebooks" / "evaluation-RQ1")
if _RQ1_DIR not in sys.path:
    sys.path.insert(0, _RQ1_DIR)

from evaluate_rq1 import (  # noqa: E402
    load_triplets, build_global_corpus, build_eval_items,
    build_binary_relevance_maps, build_sorted_pools_global,
    avg_prf_at_k_annotated, avg_mrr_at_k_annotated, avg_hits_at_k_annotated,
    avg_ndcg_binary, report,
)

RQ1_TEST_PATH = f"../../datasets/{KG}_testset_RQ1.json"
TOPKS = [3, 5, 10, 20]   # same k values as the paper tables

rq1_data = load_triplets(RQ1_TEST_PATH)
rq1_corpus, rq1_seen_q = build_global_corpus(rq1_data)
rq1_eval_items = build_eval_items(rq1_data, rq1_corpus, rq1_seen_q)
rq1_relevance_maps = build_binary_relevance_maps(rq1_eval_items, rq1_corpus)

# For DBpedia this should print: anchors=337, corpus=1677 (as in the paper).
print(f"RQ1 test set loaded from {RQ1_TEST_PATH}")
print(f"Evaluation ready -> anchors: {len(rq1_eval_items)} | corpus: {len(rq1_corpus)}")


In [ ]:
# ============================================================================
# 10b. Encode the RQ1 corpus with the fine-tuned model and rank
# ============================================================================
@torch.no_grad()
def build_emb_map_qwen3(model, queries, batch_size=16, max_length=MAX_LEN):
    """Encode raw SPARQL strings with the fine-tuned Qwen3 embedder.

    Uses the same tokenizer settings and last-token pooling as training;
    the model already returns L2-normalised float embeddings, so cosine
    similarity is a plain dot product (as evaluate_rq1 assumes).
    """
    model.eval()
    emb_map = {}
    for i in tqdm(range(0, len(queries), batch_size), desc="Encoding RQ1 corpus"):
        batch = queries[i:i + batch_size]
        tok = tokenizer(batch, padding=True, truncation=True,
                        max_length=max_length, return_tensors="pt")
        tok = {k: v.to(device) for k, v in tok.items()}
        embs = model(tok).cpu().numpy()
        for q, e in zip(batch, embs):
            emb_map[q] = e
    return emb_map


# `best_model` holds the best-val-MRR adapter reloaded in Section 9. If you
# are re-running only this section in a fresh kernel, uncomment:
# best_model = Qwen3SparqlEmbedder(MODEL_NAME, adapter_path=best_path).to(device)

rq1_emb_map = build_emb_map_qwen3(best_model, rq1_corpus, batch_size=BATCH_SIZE)
rq1_sorted_pools = build_sorted_pools_global(rq1_emb_map, rq1_eval_items, rq1_corpus)

report(f"Qwen3-Emb-0.6B (fine-tuned) — RQ1 {KG}",
       rq1_sorted_pools, rq1_eval_items, rq1_relevance_maps, TOPKS)


In [ ]:
# ============================================================================
# 10c. Collect metrics per k, emit LaTeX rows for the paper tables, save JSON
# ============================================================================
rq1_metrics = {}
for k in TOPKS:
    p, r, f1 = avg_prf_at_k_annotated(rq1_sorted_pools, rq1_eval_items, k)
    rq1_metrics[k] = {
        "MRR":  avg_mrr_at_k_annotated(rq1_sorted_pools, rq1_eval_items, k),
        "Hits": avg_hits_at_k_annotated(rq1_sorted_pools, rq1_eval_items, k),
        "nDCG": avg_ndcg_binary(rq1_sorted_pools, rq1_relevance_maps, k),
        "P": p, "R": r, "F1": f1,
    }

# Rows in the exact column order of tables/tab-RQ1-<KG>.tex
# (per k: MRR, Hits, nDCG for the ranking subtable; P, R, F1 for set-overlap).
rank_row = " & ".join(f"{rq1_metrics[k][m]:.4f}"
                      for k in TOPKS for m in ("MRR", "Hits", "nDCG"))
ovlp_row = " & ".join(f"{rq1_metrics[k][m]:.4f}"
                      for k in TOPKS for m in ("P", "R", "F1"))

print("LaTeX row — ranking-metrics subtable:")
print(f"Qwen3-Emb-0.6B (fine-tuned)~\\cite{{qwen2025qwen3}} & {rank_row} \\\\")
print("\nLaTeX row — set-overlap subtable:")
print(f"Qwen3-Emb-0.6B (fine-tuned) & {ovlp_row} \\\\")

rq1_out_path = os.path.join(CKPT_DIR, f"rq1_test_metrics_{KG}.json")
with open(rq1_out_path, "w") as f:
    json.dump({
        "kg": KG,
        "model": f"{MODEL_NAME} (fine-tuned, best adapter)",
        "protocol": "notebooks/evaluation-RQ1/evaluate_rq1.py",
        "testset": RQ1_TEST_PATH,
        "best_lr": best_lr,
        "lr_sweep": sweep_results,
        "anchors": len(rq1_eval_items),
        "corpus_size": len(rq1_corpus),
        "topks": TOPKS,
        "metrics": {str(k): rq1_metrics[k] for k in TOPKS},
    }, f, indent=2)
print(f"\nRQ1 metrics written to {rq1_out_path}")


### Notes

- **Paper numbers come from Section 10.** The RQ1 tables in the paper are
  computed on `datasets/{kg}_testset_RQ1.json` with the shared
  `evaluate_rq1.py` protocol; Section 9's pair-ranking numbers are only a
  training sanity check and are **not** comparable to the tables.
- **Comparability.** Fixed splits and the MRR-based model selection are
  identical to the SparqlCE training notebook; the in-batch negative pool
  size (32) and temperature/scale (τ=0.05 == scale=20) are matched too. The
  RQ1 evaluation reuses the exact same script as all baselines, so the
  Section 10 output slots directly into the paper's tables as the
  *fine-tuned* Qwen3 row (vs. the zero-shot Qwen3 baseline).
- **LR and epoch budget are no longer asserted.** Section 7 sweeps
  `LR_CANDIDATES` with a short probe and Section 8 early-stops on val MRR
  (patience `PATIENCE`, budget `MAX_EPOCHS`) instead of using one hand-picked
  LR and a fixed epoch count — this is what makes the recipe comparable in
  rigor to SparqlCE's own (previously tuned) hyperparameters.
- **Zero-shot reference.** To reproduce the zero-shot Qwen3 row with the same
  pipeline, skip Sections 7-8 and run Section 10 on a fresh
  `Qwen3SparqlEmbedder(MODEL_NAME, lora_cfg=lora_cfg)` — the untrained LoRA
  is an identity at init (`lora_B = 0`), so it reproduces the base model.
- **Sanity check before trusting Section 10 numbers.** A properly trained
  contrastive fine-tune should not underperform the zero-shot baseline on
  Hits/nDCG at any k — if it does, suspect a checkpoint-selection or
  pooling mismatch before reporting the numbers.
- **DBpedia. Set `KG = "dbpedia"` in the CONFIG cell; this notebook's default
  is Wikidata (`datasets/wikidata-triplets-dataset.json` /
  `splits_wikidata.json` / `datasets/wikidata_testset_RQ1.json`).**
- **Memory / compute.** ~0.6B params in bf16 + LoRA + checkpointing fits
  comfortably on a single 24 GB GPU at batch 16 / 512 tokens. Reduce
  `BATCH_SIZE` and raise `GRAD_ACCUM` proportionally if you hit OOM — in-batch
  negatives mean the *per-step* batch size also sets how many negatives each
  anchor is contrasted against. The LR sweep (Section 7) trains
  `len(LR_CANDIDATES) * LR_SWEEP_EPOCHS` extra epochs' worth of compute
  before the real run — budget for that.
